In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Seq2Seq结构

- 整体结构：

    网络结构主要由编码器和解码器两部分组成，整个应用场景是将一个样本/序列的输入转换为另外一个样本/序列的输出，比如：翻译、古诗生成、对联生成、阅读理解、关键词抽取、文本生成等；

- 编码器

    最原始的编码器一般由RNN/LSTM/GRU结构组成，输入\[bs,et]形状的原始输入token id列表，得到每个token对应的高阶特征向量以及每个文本对应的文本特征向量；bs表示一个批次中存在bs个样本，每个样本由et个token组成，__PS:每个样本实际token数目不一致，所以一个批次中可能存在填充数据；__

    案例：一个样本、5个token组成的输入数据```[[12,35,26,34,253]]```

- 解码器

    最原始的解码器一般由RNN/LSTM/GRU结构组成，输入\[bs,dt]形状的解码器token id列表，对应实际token id列表shape也是\[bs,dt]，并且解码器的输入和解码器的输出恰好有一个位置的错位，并且解码器的第一个时刻的输入和最后一个时刻的输出一般都是特殊token id；bs表示一个批次中存在bs个样本，每个样本由dt个token组成，__PS:每个样本实际token数目不一致，所以一个批次中可能存在填充数据；__

    __NOTE:解码器必须为单向结构__ 

    案例：一个样本，6个token组成的解码器数据：

        解码器输入：[[3,102,235,1523,2132,1243]]
        解码器输出：[[102,235,1523,2132,1243,4]]

In [4]:
class EncoderModule(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers):
        super().__init__()

        self.embed_layer = nn.Embedding(
            num_embeddings=vocab_size, # 词汇表大小 --> token到id转换的映射表大小
            embedding_dim=hidden_size # 每个词/Token对应的特征向量维度大小
        )
        self.rnn_layer = nn.LSTM(
            input_size = hidden_size, # 每个token输入的特征向量维度大小
            hidden_size = hidden_size,  # 每个token输出的特征向量维度大小
            num_layers = num_layers, # 层数
            batch_first = True, 
            bidirectional = True
        )
        # 将RNN输出特征值转换为高阶特征向量C
        self.ctx_feature_layer = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU()
        )

    def forward(self, token_ids):
        # 1. token id转换为token embedding向量 [bs,et] -> [bs,et,hidden_size]
        token_embed = self.embed_layer(token_ids)
        # 2. 调用rnn结构获取序列特征向量
        # output: [bs,et,2*hidden_size] --> 当前是双向结构
        # state: rnn和gru的时候，只有一个值；lstm的时候，有两个值(二元组)；shape均为[?,bs,hidden_size]
        output, state = self.rnn_layer(token_embed)
        if isinstance(state, tuple):
            state = state[0] + state[1]
        state = torch.mean(state, dim=0) # [?,bs,hidden_size] -->  [bs,hidden_size]
        # 3. 将状态信息转换为文本特征向量 [bs,hidden_size] -> [bs,hidden_size]
        ctx_embed = self.ctx_feature_layer(state)
        
        return ctx_embed

In [5]:
# 编码器案例
encoder = EncoderModule(100, 128, 3)
token_ids = torch.randint(0, 20, size=(2,5))
encoder_ctx_embed = encoder(token_ids)
print(encoder_ctx_embed.shape)

torch.Size([2, 128])


In [6]:
class DecoderModule(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers):
        super().__init__()
        self.num_layers = num_layers

        self.embed_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=hidden_size)

        self.rnn_init_h0_layers = nn.Linear(hidden_size, num_layers * hidden_size)
        self.rnn_init_c0_layers = nn.Linear(hidden_size, num_layers * hidden_size)
        
        self.rnn_layer = nn.LSTM(
            input_size = hidden_size, hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = True, bidirectional = False
        )
        self.classify_layer = nn.Linear(hidden_size, vocab_size)

    def forward(self, token_ids, encoder_ctx):
        """
            前向执行过程
            : token_ids : [bs,dt] 解码器输入token ids
            : encoder_ctx : [bs, hidden_size] 编码器提取出来的文本特征向量
        """
        # 1. 针对编码器提取特征向量进行特征提取，获取lstm的初始状态
        bs, e = encoder_ctx.shape
        h0 = self.rnn_init_h0_layers(encoder_ctx) # [bs,hidden_size] -> [bs,num_layers*hidden_size]
        h0 = h0.reshape((bs,e,-1)) # [bs,num_layers*hidden_size] --> [bs,hidden_size,num_layers]
        h0 = torch.permute(h0, dims=(2, 0, 1)) # [bs,hidden_size,num_layers] -> [num_layers,bs,hidden_size]

        c0 = self.rnn_init_c0_layers(encoder_ctx) # [bs,hidden_size] -> [bs,num_layers*hidden_size]
        c0 = c0.reshape((bs,e,-1)) # [bs,num_layers*hidden_size] --> [bs,hidden_size,num_layers]
        c0 = torch.permute(c0, dims=(2, 0, 1)) # [bs,hidden_size,num_layers] -> [num_layers,bs,hidden_size]

        if self.training:
            # 2. 针对输入数据进行embedding操作 [bs,dt] -> [bs,dt,hidden_size]
            token_embed = self.embed_layer(token_ids)
    
            # 3. 调用rnn结构获取序列特征向量
            # output: [bs,dt,hidden_size] 每个token对应的特征向量
            output, _ = self.rnn_layer(token_embed, (h0,c0))
    
            # 4. 针对每个token进行全连接得到预测对应类别 [bs,dt,hidden_size] -> [bs,dt,vocab_size]
            score = self.classify_layer(output)
            
            return score
        else:
            ##### 针对推理预测过程，需要一个token、一个token进行输入预测得到结果
            while True:
                # 2. 针对输入数据进行embedding操作 [bs,dt] -> [bs,dt,hidden_size]
                token_embed = self.embed_layer(token_ids)

                # 3. 调用rnn结构获取序列特征向量
                # output: [bs,dt,hidden_size] 每个token对应的特征向量
                output, _ = self.rnn_layer(token_embed, (h0,c0))

                # 4. 获取最后一个时刻的提取特征向量值
                output_t = output[:, -1, :]

                # 5. 预测属于各个类别的置信度
                score = self.classify_layer(output_t) # [bs, vocab_size]

                # 6. 获取预测类别id
                pred_ids = torch.argmax(score, dim=1, keepdim=True) # [bs,1] 获取置信度最大的下标作为预测类别id

                # 7. 将当前时刻的预测结果和之前的结果合并到一起
                token_ids = torch.cat([token_ids, pred_ids], dim=1) # [bs,dt+1]

                # 8. 判断是否结束生成逻辑：一般情况下至少两个条件 预测到结尾token，预测总序列长度超过一定的限制
                if token_ids.shape[1] > 10:
                    break
            return token_ids # 预测类别id

In [7]:
# 解码器案例
decoder = DecoderModule(100, 128, 3)
token_ids = torch.randint(0, 20, size=(2,5)) # 解码器的输入
encoder_ctx_embed = torch.rand(2,128) # 编码器提取出来的特征向量

decoder_score = decoder(token_ids, encoder_ctx_embed)
print(f"解码器输出:{decoder_score.shape}")

解码器输出:torch.Size([2, 5, 100])


In [8]:
class Seq2SeqModule(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers, decoder_vocab_size=None, decoder_num_layers=None):
        super().__init__()

        if decoder_vocab_size is None:
            decoder_vocab_size = vocab_size
        if decoder_num_layers is None:
            decoder_num_layers = num_layers
        # 编码器创建
        self.encoder = EncoderModule(vocab_size, hidden_size, num_layers)
        # 解码器创建
        self.decoder = DecoderModule(decoder_vocab_size, hidden_size, decoder_num_layers)

    def forward(self, encoder_token_ids, decoder_token_ids):
        # 1. 获取编码器提取出来的特征向量信息
        encoder_ctx = self.encoder(encoder_token_ids)

        # 2. 获取解码器前向执行信息
        decoder_outputs = self.decoder(decoder_token_ids, encoder_ctx)

        return decoder_outputs

In [11]:
# Seq2Seq案例

seq2seq = Seq2SeqModule(10000, 64, 3, decoder_num_layers=2)
print(seq2seq)

loss_fn = nn.CrossEntropyLoss(reduction='none')
#loss_fn = nn.CrossEntropyLoss()

Seq2SeqModule(
  (encoder): EncoderModule(
    (embed_layer): Embedding(10000, 64)
    (rnn_layer): LSTM(64, 64, num_layers=3, batch_first=True, bidirectional=True)
    (ctx_feature_layer): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
    )
  )
  (decoder): DecoderModule(
    (embed_layer): Embedding(10000, 64)
    (rnn_init_h0_layers): Linear(in_features=64, out_features=128, bias=True)
    (rnn_init_c0_layers): Linear(in_features=64, out_features=128, bias=True)
    (rnn_layer): LSTM(64, 64, num_layers=2, batch_first=True)
    (classify_layer): Linear(in_features=64, out_features=10000, bias=True)
  )
)


In [13]:
# 训练过程测试
encoder_token_ids = torch.tensor([[12,35,26,34,253]]) # [1,5]
decoder_token_ids = torch.tensor([[3,102,235,1523,2132,1243]]) # [1,6]
decoder_target_ids = torch.tensor([[102,235,1523,2132,1243,4]]) # [1,6]

seq2seq.train()
decoder_score = seq2seq(encoder_token_ids, decoder_token_ids) # [1,6,10000]
print(decoder_score.shape)

loss = loss_fn(torch.permute(decoder_score, dims=(0,2,1)), decoder_target_ids)
print(loss)

torch.Size([1, 6, 10000])
tensor([[9.2611, 9.2227, 9.3771, 9.3195, 9.2434, 9.1634]],
       grad_fn=<ViewBackward0>)


In [14]:
# 推理预测过程测试

encoder_token_ids = torch.tensor([[12,35,26,34,253]])
decoder_token_ids = torch.tensor([[3]])

seq2seq.eval()
pred_token_ids = seq2seq(encoder_token_ids, decoder_token_ids)

print(pred_token_ids.shape)
print(f"预测token id:\n\t{pred_token_ids}")

torch.Size([1, 11])
预测token id:
	tensor([[   3,  595, 2027, 2027, 4310, 2215,  595, 2215, 6286, 9048, 9048]])
